**Import bibliotek i utworzenie SparkSession**

Utworzono lokalną sesję Spark działającą w trybie local[*]. Wszystkie dostępne rdzenie procesora są wykorzystywane jako lokalny odpowiednik klastra Databricks.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("FASTQ_Analysis")
    .master("local[*]")
    .getOrCreate()
)

spark

**7.1 Wczytanie i parsowanie pliku FASTQ**

Plik FASTQ został wczytany jako DataFrame tekstowy. Każda linia pliku stanowi osobny rekord. Ponieważ pojedynczy odczyt FASTQ składa się z 4 linii, liczba odczytów została wyznaczona jako liczba wszystkich linii podzielona przez 4. Plik SRR16356247_1_1.fastq powstał poprzez wyekstrahowanie pierwszych 100 odczytów z oryginalnego pliku: 
`head -n 400 SRR16356247_1.fastq > SRR16356247_1_1.fastq`

In [ ]:
file_path = "/home/clusters/work/SRR16356247_1_1.fastq"

lines_df = spark.read.text(file_path)

lines_df.show(8, truncate=False)

print(f"Liczba linii: {lines_df.count()}")
print(f"Liczba odczytów: {lines_df.count() // 4}")

**7.2 Dodanie numerów linii**

Za pomocą funkcji `monotonically_increasing_id()` nadano każdemu wierszowi w DataFrame unikalny, rosnący numer. 

In [ ]:
lines_with_id = lines_df.withColumn(
    "line_number",
    monotonically_increasing_id()
)

lines_with_id.show(8, truncate=False)

In [ ]:
window = Window.orderBy(monotonically_increasing_id())

lines_with_id = lines_df.withColumn(
    "line_number",
    row_number().over(window) - 1
)

lines_with_id.show(8, truncate=False)

**7.3 Określenie typu linii w FASTQ**

Przypisano etykiety kolejnym wierszom na podstawie ich funkcji.

In [ ]:
lines_typed = lines_with_id.withColumn(
    "line_type",
    when(col("line_number") % 4 == 0, "header")
    .when(col("line_number") % 4 == 1, "sequence")
    .when(col("line_number") % 4 == 2, "separator")
    .when(col("line_number") % 4 == 3, "quality")
)

lines_typed.show(12, truncate=False)

**7.4 Utworzenie identyfikatora odczytu**

Nadanie wszysktim wierszom należacym do jednego rekordu tego samego `record_id`.

In [ ]:
lines_with_record = lines_typed.withColumn(
    "record_id",
    (col("line_number") / 4).cast("integer")
)

lines_with_record.show(16, truncate=False)

**7.5 Przekształcenie do formatu szerokiego (Pivot)**

In [ ]:
fastq_wide = lines_with_record.groupBy("record_id").pivot("line_type").agg(
    first("value")
)

fastq_wide.show(5, truncate=False)
fastq_wide.printSchema()

**7.6 Czyszczenie danych nagłówka**

In [ ]:
fastq_clean = fastq_wide.withColumn(
    "read_id",
    split(
        regexp_replace(col("header"), "^@", ""),
        " "
    )[0]
)

**Zadanie 1. Waidacja integralności danych**

In [ ]:
# ekstrakcja deklarowanej długości
fastq_declared = fastq_clean.withColumn(
    "declared_length",
    regexp_extract(
        col("header"),
        r"length=(\d+)",
        1
    ).cast("integer")
)

In [ ]:
# obliczenie długości rzeczywistej
fastq_actual = fastq_declared.withColumn(
    "actual_length",
    length(col("sequence"))
)

In [ ]:
# wybranie tylko potrzebnych kolumn
fastq_final = fastq_actual.select(
    "record_id",
    "read_id",
    "sequence",
    "quality",
    "declared_length",
    "actual_length"
)

fastq_final.show(5, truncate=False)

In [ ]:
#obliczenie liczby rekrodów, których dlugości się nie zgadzają
invalid_reads = fastq_final.filter(
    col("declared_length") != col("actual_length")
)

invalid_reads.count()

Joby wywołane przez fragment kodu z zadania 1:

| Zadanie (Job) | Akcja w kodzie                    | Co robi                                                                                                                           | Etapy (Stages) | Dlaczego pominięte (skipped)?                                                                           |
| ------------- | --------------------------------- | --------------------------------------------------------------------------------------------------------------------------------- | -------------- | ------------------------------------------------------------------------------------------------------- |
| 15            | `fastq_final.show(5)`             | Wykonuje transformacje tworzące kolumny `declared_length` i `actual_length`, a następnie przygotowuje 5 rekordów do wyświetlenia. | 1/1            | -                                                                                                       |
| 16            | Kontynuacja `fastq_final.show(5)` | Wykonuje dalszą część planu `show()`, wykorzystując wcześniej przygotowane dane i kończy wyświetlanie wyniku.                     | 1/1, 1 skipped | Spark pominął wcześniejszy etap odczytu danych, ponieważ został już wykonany w Job 15.                  |
| 17            | `count()`                         | Wykonuje pierwszą część planu zliczania rekordów po transformacjach. Odczytuje dane i przygotowuje je do końcowej agregacji.      | 1/1            | -                                                                                                       |
| 18            | Kontynuacja `count()`             | Wykonuje końcową agregację (`SortAggregate`) potrzebną do uzyskania wyniku `count()`.                                             | 1/1, 1 skipped | Spark pominął etap ponownego odczytu danych, ponieważ wykorzystał wcześniej przygotowane dane z Job 17. |

**Ile Jobs zostało utworzonych? Czy widzimy zależność między liczbą wywołań akcji a
liczbą zadań?**

W Spark UI widoczne są 4 Joby, mimo że w kodzie wykonano tylko dwie akcje: `show()` oraz `count()`. Wynika to z faktu, że Spark może podzielić jedną akcję na kilka Jobów zależnie od planu wykonania zapytania. Dlatego liczba Jobów nie musi być równa liczbie wywołanych akcji. W tym przypadku akcje `show()` i `count()` wygenerowały po dwa Joby, ponieważ Spark optymalizował i dzielił wykonanie operacji zawierających wcześniejsze transformacje i agregacje.

**Które operacje były transformacjami, a które akcjami? Wypiszmy je, aby utrwalić tę
kluczową różnicę.**

**Transformacje:**

* withColumn()
* select()
* length()
* regexp_extract()
* filter()

**Akcje:**

* show()
* count()


**Sprawdźmy DAG Visualization dla tego Joba. Ile etapów (RDD/DataFrame) tworzy plan
wykonania zapytania?**


DAG Visualization dla Job 17 pokazuje jeden wykonany Stage. Plan wykonania obejmuje odczyt danych oraz wcześniejsze transformacje prowadzące do obliczenia wyniku `count()`. Brak dodatkowych etapów wynika z tego, że operacja zliczania rekordów nie wymaga wymiany danych między partycjami (shuffle).

In [ ]:
fastq_final.rdd.getNumPartitions()

**Ile tasków wykonał jedyny Stage w tym Jobie? Porównajmy tę liczbę z domyślną liczbą
partycji DataFrame. Wyciągnijmy wniosek o relacji między partycjami a taskami.**

Jedyny Stage w Job 17 wykonał 1 task. Task jest najmniejszą jednostką pracy w Spark i przetwarza jedną partycję danych. Sprawdzając fastq_final.rdd.getNumPartitions() otrzymano wynik 1, co oznacza, że DataFrame miał jedną partycję. Jest to zgodne z niewielkim rozmiarem pliku (35,5 KiB), który był znacznie mniejszy od domyślnego maksymalnego rozmiaru partycji (128 MB). Liczba partycji odpowiada, więc liczbie tasków.